# Embedding CLIP cho keyframe (tuỳ chọn)

Backend hiện chỉ dùng SigLIP-2 và BEiT-3; notebook này để thử nghiệm model khác.

- **Đầu vào:** `<keyframe-dir>/<video_id>/...` (dataset Kaggle).
- **Đầu ra:** `/kaggle/working/clip/<video_id>.npy`, dòng *i* là ảnh thứ *i* theo thứ tự đường dẫn,
  chuẩn hoá L2. Ảnh lỗi làm dừng video đó thay vì thành vector ảnh đen.

In [ ]:
!pip install -q "transformers>=4.56"

## Cấu hình

In [ ]:
from pathlib import Path

KEYFRAME_DIR = Path("/kaggle/input/aic-2026-keyframes/keyframes")
OUTPUT_DIR = Path("/kaggle/working/clip")
MODEL_ID = "openai/clip-vit-base-patch32"

SHARD_INDEX = 1  # ← đổi theo máy: 1, 2, 3 hoặc 4
NUM_SHARDS = 4

BATCH_SIZE = 128
NUM_WORKERS = 2

## Trích embedding

In [ ]:
import numpy as np
import torch
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from transformers import CLIPModel, CLIPProcessor


def select_shard(items, shard_index, num_shards, weight=None):
    """Part shard_index (1-based) of num_shards. Items are sorted first so every machine agrees.

    Without weight: contiguous chunks. With weight: the heaviest item goes to the lightest
    shard (e.g. by video size), so all machines finish at about the same time.
    """
    assert 1 <= shard_index <= num_shards, "SHARD_INDEX phải nằm trong [1, NUM_SHARDS]"
    ordered = sorted(items)
    if num_shards == 1:
        return ordered
    if weight is None:
        size = -(-len(ordered) // num_shards)
        return ordered[(shard_index - 1) * size : shard_index * size]
    shards = [[] for _ in range(num_shards)]
    totals = [0.0] * num_shards
    for item in sorted(ordered, key=weight, reverse=True):
        lightest = totals.index(min(totals))
        shards[lightest].append(item)
        totals[lightest] += weight(item)
    return sorted(shards[shard_index - 1])


def matches_any(video_id, selectors):
    """True when no selector is given or video_id equals / starts with one of them."""
    return not selectors or any(video_id == s or video_id.startswith(s) for s in selectors)


class KeyframeDataset(Dataset):
    def __init__(self, image_paths, processor):
        self.image_paths = image_paths
        self.processor = processor

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        with Image.open(self.image_paths[idx]) as image:
            return self.processor(images=image.convert("RGB"), return_tensors="pt").pixel_values[0]


device = "cuda" if torch.cuda.is_available() else "cpu"
processor = CLIPProcessor.from_pretrained(MODEL_ID)
model = CLIPModel.from_pretrained(MODEL_ID).to(device).eval()

video_dirs = select_shard(
    [p for p in KEYFRAME_DIR.iterdir() if p.is_dir()], SHARD_INDEX, NUM_SHARDS
)
print(f"Shard {SHARD_INDEX}/{NUM_SHARDS}: {len(video_dirs)} video")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
failed = {}
with torch.inference_mode():
    for video_dir in tqdm(video_dirs, desc="CLIP"):
        output_file = OUTPUT_DIR / f"{video_dir.name}.npy"
        if output_file.exists():
            continue
        keyframes = sorted(
            p for p in video_dir.rglob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png"}
        )
        if not keyframes:
            continue
        loader = DataLoader(
            KeyframeDataset(keyframes, processor), batch_size=BATCH_SIZE, num_workers=NUM_WORKERS
        )
        try:
            chunks = []
            for pixel_values in loader:
                features = model.get_image_features(pixel_values=pixel_values.to(device))
                chunks.append(torch.nn.functional.normalize(features, dim=-1).cpu().numpy())
        except OSError as exc:
            failed[video_dir.name] = str(exc)
            continue
        np.save(output_file, np.concatenate(chunks).astype(np.float32))

for video_id, error in failed.items():
    print(f"[LỖI] {video_id}: {error}")

## Nén kết quả
Tải file nén ở tab **Output** của Kaggle rồi giải nén vào `data/embeddings/clip/` trên máy build.

In [ ]:
import shutil

archive = shutil.make_archive(
    str(OUTPUT_DIR.parent / f"{OUTPUT_DIR.name}_shard{SHARD_INDEX}of{NUM_SHARDS}"),
    "zip",
    root_dir=str(OUTPUT_DIR.parent),
    base_dir=OUTPUT_DIR.name,
)
print(f"{archive}: {Path(archive).stat().st_size / 2**20:.1f} MB")